# LVK: compact-binary parameter estimation

**FQCP 2026 · Bayesian parameter estimation for gravitational-wave sources**

> Google Colab worksheet for early-stage graduate students. Run from top to
> bottom. In the JupyterBook, **Live route** cards identify the material for the
> session; **Extension** sections may be skipped live.

## Goal and analysis map

Follow a compact version of the NZ Bilby workshop's full CBC flow:

$$
\theta_{\rm CBC}\rightarrow(h_+,h_\times)\rightarrow h_I
\rightarrow d_I=h_I+n_I\rightarrow\mathcal L_{\rm network}\rightarrow p(\theta\mid d).
$$

We use rippleGW for an actual IMRPhenomD waveform and Bilby for detector
geometry, PSDs, projection, and injection. Readable grid posteriors replace a
slow live sampler. The aim is to make every layer visible before a high-level
library combines them.

Section 3 adds the step that comes before any parameter estimation: **matched
filtering**, the search stage that finds the signal and produces the trigger.
Section 4 then estimates parameters from it, including a two-dimensional
posterior that shows the distance-inclination degeneracy behind
gravitational-wave distance uncertainties.

:::{admonition} Live route — 45 minutes
:class: tip

Use the dropdowns immediately below to separate the in-room sequence from the
reference material.
:::

:::{dropdown} In the room
Sections 1–5: CBC parameters, detector response, matched filtering, the manual
likelihood, and network localisation. Run the distance–inclination posterior.
:::
:::{dropdown} Read afterwards
Section 6 is a compact population-inference bridge; Section 7 is a genuine
Bilby/dynesty run and can take a few minutes in Colab.
:::

:::{admonition} Do not collapse these three questions
:class: important

| Task | Question | Typical output |
| --- | --- | --- |
| detection/search | Is there a candidate inconsistent with noise? | trigger, ranking statistic, false-alarm control |
| parameter estimation | Which source parameters remain plausible? | posterior, credible intervals, posterior predictions |
| population inference | What generated many detected events? | hyperposterior, selection-aware population model |

Matched filtering in Section 3 is the **search** bridge. The likelihood in
Section 4 begins **parameter estimation**. Section 6 asks the population
question and must account for what the detectors were able to find.
:::

In [ ]:
import os, sys, subprocess, importlib.util

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
missing = [
    p for p in ("ripplegw", "bilby", "gwpy") if importlib.util.find_spec(p) is None
]
if missing:
    if IN_COLAB:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "rippleGW==0.2.1",
                "bilby==2.8.0",
                "gwpy>=3.0,<4",
            ]
        )
    else:
        raise ImportError(
            "Install rippleGW==0.2.1, bilby==2.8.0, and gwpy>=3.0,<4, or run in Colab."
        )

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt
import bilby
from gwpy.timeseries import TimeSeries
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from jax import config

config.update("jax_enable_x64", True)
import jax.numpy as jnp
from ripplegw.conversions import ms_to_Mc_eta
from ripplegw.waveforms.IMRPhenomD import gen_IMRPhenomD_hphc

logging.getLogger("bilby").setLevel(logging.ERROR)
plt.style.use("seaborn-v0_8-whitegrid")


def show_animation(animation):
    # H.264 video: ~40x smaller in the notebook than one PNG per frame
    try:
        return display(HTML(animation.to_html5_video()))
    except RuntimeError:  # ffmpeg unavailable: fall back to per-frame PNGs
        return display(HTML(animation.to_jshtml()))


rng = np.random.default_rng(20260817)

from matplotlib.ticker import NullFormatter, ScalarFormatter


def tidy_log_frequency(axis, ticks=(20, 50, 100, 200, 500)):
    """Readable Hz labels: matplotlib's log minor ticks overlap on wide bands."""
    axis.set_xticks(list(ticks))
    axis.xaxis.set_major_formatter(ScalarFormatter())
    axis.xaxis.set_minor_formatter(NullFormatter())

## 1. CBC parameters

| Group | Examples | Main effect |
| --- | --- | --- |
| masses | $m_1,m_2$ or chirp mass $\mathcal M$ and $q=m_2/m_1$ | phase and merger frequency |
| spins | magnitudes and orientations | phase, precession, merger |
| matter/orbit | tides, eccentricity | extra phase and harmonics |
| location | right ascension, declination, distance | detector response and amplitude |
| orientation | inclination $\iota$, polarisation $\psi$, phase | relative polarisation content |
| time | geocentric coalescence time | detector arrival times |

$$
\mathcal M=\frac{(m_1m_2)^{3/5}}{(m_1+m_2)^{1/5}},\qquad
m_{\rm detector}=(1+z)m_{\rm source}.
$$

Intrinsic/extrinsic is useful bookkeeping, but parameters remain correlated in the posterior.

In [ ]:
sample_rate, duration, f_min = 1024, 4, 20.0
gps_time = 1126259462.4
frequency = np.fft.rfftfreq(int(sample_rate * duration), 1 / sample_rate)
mask = frequency >= f_min
df = frequency[1] - frequency[0]


def ripple_parameters(
    chirp_mass=None,
    m1=36.0,
    m2=29.0,
    chi1=0.1,
    chi2=-0.1,
    distance=800.0,
    tc=0.0,
    phase=0.3,
    inclination=0.5,
):
    mc, eta = ms_to_Mc_eta(jnp.array([m1, m2]))
    mc = mc if chirp_mass is None else chirp_mass
    return jnp.array([mc, eta, chi1, chi2, distance, tc, phase, inclination])


def polarizations(theta):
    hp, hx = gen_IMRPhenomD_hphc(jnp.asarray(frequency[mask]), theta, f_min)
    result = {
        "plus": np.zeros(frequency.size, dtype=complex),
        "cross": np.zeros(frequency.size, dtype=complex),
    }
    result["plus"][mask] = np.asarray(hp)
    result["cross"][mask] = np.asarray(hx)
    return result


theta_true = ripple_parameters()
injection_polarizations = polarizations(theta_true)
print(f"Detector-frame chirp mass: {float(theta_true[0]):.3f} solar masses")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for name, h in injection_polarizations.items():
    axes[0].loglog(frequency[mask], np.abs(h[mask]), label=name)
axes[0].set(
    xlabel="frequency [Hz]",
    ylabel="strain / Hz",
    title="Radiation has two polarisations",
)
axes[0].legend()
axes[1].semilogx(
    frequency[mask], np.unwrap(np.angle(injection_polarizations["plus"][mask]))
)
axes[1].set(
    xlabel="frequency [Hz]",
    ylabel="phase [rad]",
    title="Hundreds of radians accumulate in band",
)
for ax in axes:
    tidy_log_frequency(ax)
plt.show()

For a non-precessing circular binary, approximately
$h_+\propto(1+\cos^2\iota)/(2D_L)$ and $h_\times\propto\cos\iota/D_L$.
Inclination is the binary's orientation to us; polarisation angle rotates the plus/cross basis on the sky.

### Animation: why chirp mass is measured so precisely

The signal accumulates hundreds of radians of phase in band, so a wrong chirp
mass shows up as **dephasing**, not as a wrong amplitude.

A template is free to slide in time and shift its overall phase, and matched
filtering maximises over both. So the fair comparison removes a linear-in-$f$
term first:

$$
\Delta\psi(f)=\psi_{\mathcal M}(f)-\psi_{\mathcal M_{\rm true}}(f)
-\underbrace{(a f+b)}_{\text{absorbed by }t_c,\ \phi_c}.
$$

What is left cannot be absorbed and is what destroys the match.

- Left: the amplitude barely moves. You could not measure $\mathcal M$ this way.
- Right: the residual dephasing. Once $|\Delta\psi|$ exceeds about 1 radian
  (grey band) the template and signal drift out of step and the recovered SNR
  falls, as Section 3 will show directly.

In [ ]:
mass_offsets = np.linspace(-2.0, 2.0, 21)
mc_true = float(theta_true[0])
f_band = frequency[mask]
reference = injection_polarizations["plus"][mask]
reference_phase = np.unwrap(np.angle(reference))
# Weight the tc/phi_c fit by signal power so it reflects where the SNR is.
weight = np.abs(reference) ** 2
basis = np.vstack([f_band, np.ones_like(f_band)]).T
weighted_basis = basis * np.sqrt(weight)[:, None]

fig, (amp_ax, phase_ax) = plt.subplots(1, 2, figsize=(11, 3.5), dpi=80)
amp_ax.loglog(f_band, np.abs(reference), color="0.7", lw=3, label="injection")
(amp_line,) = amp_ax.loglog([], [], color="C0", label="trial template")
amp_ax.set(
    xlim=(20, 512),
    ylim=(1e-25, 3e-22),
    xlabel="frequency [Hz]",
    ylabel=r"$|h_+|$",
    title="amplitude: almost no information",
)
amp_ax.legend(loc="lower left", fontsize=8)

phase_ax.axhspan(-1, 1, color="0.8", alpha=0.7)
phase_ax.axhline(0, color="k", lw=0.8)
(phase_line,) = phase_ax.semilogx([], [], color="C3")
phase_ax.set(
    xlim=(20, 512),
    ylim=(-10, 10),
    xlabel="frequency [Hz]",
    ylabel=r"$\Delta\psi$ [rad]",
    title="residual dephasing: all the information",
)
for ax in (amp_ax, phase_ax):
    tidy_log_frequency(ax)
fig.subplots_adjust(top=0.80, wspace=0.28)


def animate_mass(i):
    trial = polarizations(theta_true.at[0].set(mc_true + mass_offsets[i]))["plus"][mask]
    difference = np.unwrap(np.angle(trial)) - reference_phase
    coefficients = np.linalg.lstsq(
        weighted_basis, difference * np.sqrt(weight), rcond=None
    )[0]
    residual = difference - basis @ coefficients
    amp_line.set_data(f_band, np.abs(trial))
    phase_line.set_data(f_band, residual)
    fig.suptitle(
        f"chirp mass error {mass_offsets[i]:+.2f} solar masses "
        f"({100 * mass_offsets[i] / mc_true:+.1f}%), "
        f"peak dephasing {np.abs(residual).max():.1f} rad"
    )
    return amp_line, phase_line


mass_animation = FuncAnimation(
    fig, animate_mass, frames=len(mass_offsets), interval=160
)
plt.close(fig)
show_animation(mass_animation)

### Fast inspiral cartoon—physics intuition, not numerical relativity

This deliberately cheap animation connects orbital motion to a chirping quadrupole signal. It is not a surrogate or merger-remnant prediction: the actual waveform animation above is the quantitative one.

In [ ]:
cartoon_time = np.linspace(0, 1, 40)
radius = 1 - 0.82 * cartoon_time
orbital_phase = 2 * np.pi * (1.3 * cartoon_time + 5 * cartoon_time**3)
x = radius * np.cos(orbital_phase)
y = radius * np.sin(orbital_phase)
cartoon_strain = (1 / radius) * np.cos(2 * orbital_phase)
cartoon_strain /= np.max(np.abs(cartoon_strain))
fig, (orbit_ax, strain_ax) = plt.subplots(1, 2, figsize=(10, 4))
(body_1,) = orbit_ax.plot([], [], "o", ms=9, color="C0")
(body_2,) = orbit_ax.plot([], [], "o", ms=7, color="C1")
(separation,) = orbit_ax.plot([], [], color="0.6")
(strain_line,) = strain_ax.plot([], [], color="C3")
(marker,) = strain_ax.plot([], [], "o", color="C3")
orbit_ax.set(
    xlim=(-1.1, 1.1),
    ylim=(-1.1, 1.1),
    aspect="equal",
    xlabel="x [cartoon]",
    ylabel="y [cartoon]",
    title="shrinking, accelerating orbit",
)
strain_ax.set(
    xlim=(0, 1),
    ylim=(-1.1, 1.1),
    xlabel="time to merger [cartoon]",
    ylabel="normalised strain",
    title="frequency and amplitude increase",
)


def animate_inspiral(i):
    body_1.set_data([x[i]], [y[i]])
    body_2.set_data([-x[i]], [-y[i]])
    separation.set_data([-x[i], x[i]], [-y[i], y[i]])
    strain_line.set_data(cartoon_time[: i + 1], cartoon_strain[: i + 1])
    marker.set_data([cartoon_time[i]], [cartoon_strain[i]])
    return body_1, body_2, separation, strain_line, marker


inspiral_animation = FuncAnimation(
    fig, animate_inspiral, frames=len(cartoon_time), interval=55
)
plt.close(fig)
show_animation(inspiral_animation)

## 2. From source to a detector network

Each detector sees one projected combination of the two polarisations, delayed
by its own light-travel time:

$$
\tilde h_I=\left[F^I_+(\alpha,\delta,\psi,t)\,h_+
+F^I_\times(\alpha,\delta,\psi,t)\,h_\times\right]e^{-2\pi if\Delta t_I}.
$$

If the detector noises are independent given their PSDs, the network likelihood
is a product, so log-likelihoods simply add:

$$
\log\mathcal L_{\rm net}=\sum_I\log\mathcal L_I
=-\frac12\sum_I(d_I-h_I\mid d_I-h_I)_I+C.
$$

- $F_+^I,F_\times^I$ depend on sky position, polarisation, detector
  orientation, and sidereal time. Bilby stores the geometry and applies this.
- $\Delta t_I$ is the arrival-time delay, and differences between detectors are
  what localise the source (Section 5).
- **Source parameters are shared**; only the response and the noise weighting
  are detector-specific. That is the whole reason a network beats one detector.
- Adding a detector adds its $(d\mid h)$ terms, so SNRs add in quadrature.

In [ ]:
source_parameters = dict(ra=1.2, dec=-0.4, psi=0.7, geocent_time=gps_time)
ifos = bilby.gw.detector.InterferometerList(["H1", "L1", "V1"])
for ifo in ifos:
    ifo.set_strain_data_from_zero_noise(
        sampling_frequency=sample_rate, duration=duration, start_time=gps_time - 2
    )
print("IFO     F+      Fx      delay [ms]")
for ifo in ifos:
    fp = ifo.antenna_response(
        source_parameters["ra"],
        source_parameters["dec"],
        gps_time,
        source_parameters["psi"],
        "plus",
    )
    fx = ifo.antenna_response(
        source_parameters["ra"],
        source_parameters["dec"],
        gps_time,
        source_parameters["psi"],
        "cross",
    )
    dt = ifo.time_delay_from_geocenter(
        source_parameters["ra"], source_parameters["dec"], gps_time
    )
    print(f"{ifo.name:>3}  {fp:+.3f}  {fx:+.3f}   {1e3*dt:+.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ifo in ifos:
    asd = ifo.power_spectral_density.get_amplitude_spectral_density_array(frequency)
    axes[0].loglog(frequency[mask], asd[mask], label=ifo.name)
    response = ifo.get_detector_response(
        injection_polarizations, source_parameters, frequencies=frequency
    )
    axes[1].loglog(frequency[mask], np.abs(response[mask]), label=ifo.name)
axes[0].set(
    xlabel="frequency [Hz]",
    ylabel=r"ASD [1/$\sqrt{\mathrm{Hz}}$]",
    title="Each detector has a PSD",
)
axes[1].set(
    xlabel="frequency [Hz]",
    ylabel="projected strain / Hz",
    title="Each detector sees a different signal",
)
for ax in axes:
    tidy_log_frequency(ax)
    ax.legend()
plt.show()

### All-sky network response: predict, then inspect

**Predict before running:** Does adding a detector make every sky direction
equally loud, or does it mainly fill particular blind spots? Which information
needed for localisation is absent from a sensitivity map?

Each detector's polarisation-averaged sensitivity to a direction is

$$
R_I(\alpha,\delta)=\sqrt{F_+^{I\,2}+F_\times^{I\,2}},\qquad
\text{network proxy}=\sqrt{\sum_I\left(\frac{R_I}{\mathrm{ASD}_I}\right)^2}.
$$

- $R_I$ is **independent of the polarisation angle** $\psi$: rotating $\psi$
  mixes $F_+$ and $F_\times$ but preserves this combination.
- A single interferometer has a quadrupolar pattern with four blind spots.
  Watch how the three individual maps put their blind spots in *different*
  places, so the network map is far more uniform.
- The proxy is noise-weighted with each ASD at 100 Hz. It is not a real SNR:
  that also needs the waveform, distance, inclination, and full PSD.
- A sensitivity map says how *loud* a source is, not *where* it is. Localisation
  comes from arrival-time and phase differences, which Section 5 covers.

In [ ]:
sky_ra = np.linspace(-np.pi, np.pi, 73)
sky_dec = np.linspace(-np.pi / 2, np.pi / 2, 37)
sky_ra_grid, sky_dec_grid = np.meshgrid(sky_ra, sky_dec)
reference_frequency = 100.0
reference_index = np.argmin(np.abs(frequency - reference_frequency))
asd_reference = np.array(
    [
        ifo.power_spectral_density.get_amplitude_spectral_density_array(frequency)[
            reference_index
        ]
        for ifo in ifos
    ]
)


def response_and_snr_proxy(ra, dec):
    responses = []
    for ifo in ifos:
        f_plus = ifo.antenna_response(
            ra, dec, gps_time, source_parameters["psi"], "plus"
        )
        f_cross = ifo.antenna_response(
            ra, dec, gps_time, source_parameters["psi"], "cross"
        )
        responses.append(np.hypot(f_plus, f_cross))
    responses = np.asarray(responses)
    return responses, np.sqrt(np.sum((responses / asd_reference) ** 2, axis=0))


# Per-detector response maps and the noise-weighted network proxy.
detector_maps = np.array(
    [
        [[response_and_snr_proxy(ra, dec)[0][k] for ra in sky_ra] for dec in sky_dec]
        for k in range(len(ifos))
    ]
)
snr_proxy_map = np.array(
    [[response_and_snr_proxy(ra, dec)[1] for ra in sky_ra] for dec in sky_dec]
)
snr_proxy_scale = snr_proxy_map.max()
detector_names = [ifo.name for ifo in ifos]

detector_scales = detector_maps.max(axis=(1, 2))
panels = [
    (name, detector_maps[k] / detector_scales[k])
    for k, name in enumerate(detector_names)
]
panels.append(("network", snr_proxy_map / snr_proxy_scale))

# Four reasonably sized maps reveal the blind spots more clearly than one
# compressed row. The bar chart answers the local question at the marker.
fig = plt.figure(figsize=(12, 6.8), dpi=78)
grid_spec = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.72], hspace=0.24, wspace=0.16)
markers = []
map_axes = []
for panel_index, (name, field) in enumerate(panels):
    row, column = divmod(panel_index, 2)
    sky_ax = fig.add_subplot(grid_spec[row, column], projection="mollweide")
    map_axes.append(sky_ax)
    image = sky_ax.pcolormesh(
        sky_ra_grid, sky_dec_grid, field, shading="auto", cmap="viridis", vmin=0, vmax=1
    )
    (marker,) = sky_ax.plot([], [], "o", color="C3", mec="white", ms=7)
    markers.append(marker)
    sky_ax.grid(True, lw=0.4, alpha=0.5)
    sky_ax.set_xticklabels([])
    sky_ax.set_yticklabels([])
    sky_ax.set_title(
        f"{name} blind spots" if name != "network" else "network (noise-weighted)",
        fontsize=10,
    )
fig.colorbar(
    image,
    ax=map_axes,
    location="bottom",
    pad=0.08,
    shrink=0.72,
    label="normalised response",
)

sky_ra_frames = np.linspace(-np.pi, np.pi, 24, endpoint=False)
response_ax = fig.add_subplot(grid_spec[:, 2])
bars = response_ax.bar(
    [*detector_names, "network"],
    np.zeros(4),
    color=["C0", "C1", "C2", "0.25"],
)
response_ax.set(
    ylim=(0, 1.05),
    ylabel="normalised response at marker",
    title="what this sky position gives",
)
response_ax.tick_params(axis="x", rotation=30)


def animate_sky_response(frame):
    source_ra = sky_ra_frames[frame]
    for marker in markers:
        marker.set_data([source_ra], [source_parameters["dec"]])
    responses, network_response = response_and_snr_proxy(
        source_ra, source_parameters["dec"]
    )
    values = np.r_[responses / detector_scales, network_response / snr_proxy_scale]
    for bar, value in zip(bars, values):
        bar.set_height(value)
    fig.suptitle(
        f"source at right ascension {source_ra:+.2f} rad; "
        f"network proxy {values[-1]:.2f}"
    )
    return (*markers, *bars)


response_animation = FuncAnimation(
    fig, animate_sky_response, frames=len(sky_ra_frames), interval=140
)
plt.close(fig)
show_animation(response_animation)

## 3. Finding the signal first: matched filtering

Before anyone estimates parameters, something has to notice that a signal is
there. In real strain data a loud binary is still far below the noise: the
whitened signal peaks at a few tenths of the noise standard deviation, so no
amount of staring at the time series will show it.

The optimal linear filter for a *known* waveform in Gaussian noise is the
matched filter. Slide a normalised template through the data and record the
overlap as a function of trial coalescence time $\tau$:

$$
z(\tau)=\frac{(d\mid h_\tau)}{\sqrt{(h\mid h)}},\qquad
h_\tau(f)=h(f)\,e^{-2\pi i f\tau}.
$$

Because the time shift is only a phase ramp in frequency, the whole SNR time
series comes from a single inverse FFT rather than one integral per trial time.
Two numbers matter, and they are not the same:

- the **optimal SNR** $\rho_{\rm opt}=\sqrt{(h\mid h)}$, what a perfect template
  would achieve on average;
- the **matched-filter SNR**, the value actually recovered, which scatters about
  $\rho_{\rm opt}$ and is biased high at the peak because we maximised over
  $\tau$.

Searches repeat this over a bank of $\sim10^6$ templates. Parameter estimation
then starts from the resulting trigger.

In [ ]:
bilby.core.utils.random.seed(2026)
noisy_ifos = bilby.gw.detector.InterferometerList(["H1", "L1", "V1"])
for ifo in noisy_ifos:
    ifo.set_strain_data_from_power_spectral_density(
        sampling_frequency=sample_rate, duration=duration, start_time=gps_time - 2
    )
    ifo.inject_signal_from_waveform_polarizations(
        source_parameters, injection_polarizations
    )

n_samples = int(sample_rate * duration)
segment_time = np.arange(n_samples) / sample_rate  # seconds after segment start
# fftshift puts zero trial offset in the middle: bilby already places the
# merger at geocent_time, so the peak should land at an offset of zero.
trial_offset = (np.arange(n_samples) - n_samples // 2) / sample_rate


def matched_filter(ifo, template_polarizations):
    """Return the SNR time series and the optimal SNR for one detector."""
    template = ifo.get_detector_response(
        template_polarizations, source_parameters, frequencies=frequency
    )
    psd = ifo.power_spectral_density_array
    usable = mask & np.isfinite(psd) & (psd > 0)
    integrand = np.zeros(frequency.size, dtype=complex)
    integrand[usable] = (
        ifo.frequency_domain_strain[usable] * np.conj(template[usable]) / psd[usable]
    )
    padded = np.zeros(n_samples, dtype=complex)
    padded[: integrand.size] = integrand
    z = 4 * df * n_samples * np.fft.ifft(padded)
    optimal = np.sqrt(4 * df * np.sum(np.abs(template[usable]) ** 2 / psd[usable]))
    return np.fft.fftshift(np.abs(z)) / optimal, optimal


fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ifo in noisy_ifos:
    snr_series, optimal_snr = matched_filter(ifo, injection_polarizations)
    peak = np.argmax(snr_series)
    axes[0].plot(trial_offset, snr_series, lw=0.7, label=ifo.name)
    axes[1].plot(trial_offset, snr_series, lw=1.2, label=ifo.name)
    print(
        f"{ifo.name}: optimal SNR {optimal_snr:5.2f} | "
        f"recovered peak {snr_series[peak]:5.2f} at "
        f"{trial_offset[peak]:+.4f} s"
    )
axes[0].set(
    xlabel="trial coalescence time offset [s]",
    ylabel=r"$|z(\tau)|$",
    title="Matched-filter SNR across the whole segment",
)
axes[1].set(
    xlim=(-0.05, 0.05),
    xlabel="trial coalescence time offset [s]",
    title="Zoom: the trigger is sharply localised in time",
)
for ax in axes:
    ax.legend()
plt.show()

### Animation: sliding the template through whitened data

- Whitening divides each Fourier bin by the noise ASD, so every frequency
  carries comparable noise. This is the weighting the likelihood applies.
- Left: the whitened template slides across the whitened H1 data.
  Right: the SNR that the overlap produces at each shift.
- The signal is invisible by eye, yet the filter finds it: the template adds
  the signal **coherently** over hundreds of cycles while noise adds
  incoherently. That is the $\sqrt{N_{\rm cycles}}$ gain.

**Predict before running:** Why can the filter find a signal that is
invisible by eye in the whitened data, and why would an incorrect phase
evolution stop that coherent accumulation?

In [ ]:
def whiten(frequency_series, psd):
    usable = mask & np.isfinite(psd) & (psd > 0)
    whitened = np.zeros(frequency.size, dtype=complex)
    whitened[usable] = frequency_series[usable] / np.sqrt(psd[usable] / (4 * df))
    return np.fft.irfft(whitened, n=n_samples)


h1 = noisy_ifos[0]
h1_psd = h1.power_spectral_density_array
whitened_data = whiten(h1.frequency_domain_strain, h1_psd)
whitened_template = whiten(
    h1.get_detector_response(
        injection_polarizations, source_parameters, frequencies=frequency
    ),
    h1_psd,
)
h1_snr_series, _ = matched_filter(h1, injection_polarizations)

lags = np.linspace(-0.3, 0.3, 45)
window = (segment_time > 1.3) & (segment_time < 2.35)

fig, (data_ax, snr_ax) = plt.subplots(1, 2, figsize=(10.5, 3.6), dpi=72)
data_ax.plot(segment_time[window], whitened_data[window], lw=0.6, color="0.55")
(template_line,) = data_ax.plot([], [], lw=1.4, color="C3")
data_ax.set(
    xlabel="time after segment start [s]",
    ylabel="whitened strain",
    title="whitened H1 data (grey) and trial template (red)",
)
(snr_trace,) = snr_ax.plot([], [], color="C0")
(snr_head,) = snr_ax.plot([], [], "o", color="C3")
snr_ax.set(
    xlim=(lags[0], lags[-1]),
    ylim=(0, 1.1 * h1_snr_series.max()),
    xlabel="template time shift [s]",
    ylabel=r"$|z(\tau)|$",
    title="overlap accumulated by the filter",
)
fig.subplots_adjust(top=0.78, wspace=0.28)


def animate_filter(i):
    shift = int(round(lags[i] * sample_rate))
    shifted = np.roll(whitened_template, shift)
    template_line.set_data(segment_time[window], shifted[window])
    used = trial_offset <= lags[i]
    snr_trace.set_data(trial_offset[used], h1_snr_series[used])
    snr_value = np.interp(lags[i], trial_offset, h1_snr_series)
    snr_head.set_data([lags[i]], [snr_value])
    fig.suptitle(f"time shift {lags[i]:+.3f} s, SNR {snr_value:.1f}")
    return template_line, snr_trace, snr_head


filter_animation = FuncAnimation(fig, animate_filter, frames=len(lags), interval=110)
plt.close(fig)
show_animation(filter_animation)

### A template only works if it is close enough

- A search cannot use the true waveform: the true parameters are what we are
  looking for. It uses a **bank** of templates and hopes one is close enough.
- Below, the same data are filtered with deliberately wrong chirp masses.
- How fast the recovered SNR falls sets how densely the bank must be packed.
  Banks are built to lose no more than a few percent of SNR anywhere.
- Compare with the dephasing animation in Section 1: the SNR loss here is that
  dephasing, integrated over the band.

### Code studio: build a tiny template bank

Write a function that loops over chirp-mass offsets, builds each trial waveform,
runs `matched_filter`, and stores the largest recovered SNR. Use only the
objects already defined above. The peak should lie close to zero offset.

In [ ]:
def student_template_bank_scan(offsets):
    # YOUR CODE HERE
    return None


student_bank = student_template_bank_scan(np.linspace(-2, 2, 9))
if student_bank is None:
    print("Your turn: return one peak SNR for every trial chirp-mass offset.")
else:
    student_bank = np.asarray(student_bank)
    assert student_bank.shape == (9,)
    best_offset = np.linspace(-2, 2, 9)[np.argmax(student_bank)]
    assert abs(best_offset) <= 0.5
    print(f"check passed; best template offset = {best_offset:+.1f} solar masses")

<details>
<summary>Show one possible solution</summary>

```python
def student_template_bank_scan(offsets):
    peaks = []
    for offset in offsets:
        trial = polarizations(theta_true.at[0].set(float(theta_true[0]) + offset))
        snr_series, _ = matched_filter(h1, trial)
        peaks.append(snr_series.max())
    return np.asarray(peaks)
```

</details>

In [ ]:
mismatch_offsets = np.linspace(-4, 4, 25)
recovered_peaks = []
for offset in mismatch_offsets:
    trial = polarizations(theta_true.at[0].set(float(theta_true[0]) + offset))
    snr_series, _ = matched_filter(h1, trial)
    recovered_peaks.append(snr_series.max())

fig, ax = plt.subplots(figsize=(7.5, 3.3))
ax.plot(mismatch_offsets, recovered_peaks, "o-")
ax.axvline(0, color="k", ls="--", label="true chirp mass")
ax.set(
    xlabel="chirp-mass error of the template [solar masses]",
    ylabel="recovered peak SNR",
    title="Template mismatch loses signal-to-noise",
)
ax.legend()
plt.show()

## 4. Inject and infer manually

We free only the detector-frame chirp mass $\mathcal M$:

$$
p(\mathcal M\mid d)\propto \pi(\mathcal M)
\exp\!\left[-\frac12\sum_I
(d_I-h_I(\mathcal M)\mid d_I-h_I(\mathcal M))_I\right].
$$

- The waveform changes once, then is projected into H1, L1, and Virgo.
- Independent detector log likelihoods add to form the network likelihood.
- We use zero-noise data so the width is deterministic; the PSD still sets the
  expected uncertainty.
- The grid is deliberately zoomed to $\pm0.1\,M_\odot$. On the old wide scale
  both posteriors were visually indistinguishable spikes.

Replace `set_strain_data_from_zero_noise` with Bilby's PSD-noise method to study
scatter across noise realisations.

In [ ]:
for ifo in ifos:
    ifo.inject_signal_from_waveform_polarizations(
        source_parameters, injection_polarizations
    )
print(
    "Network optimal SNR:",
    round(np.sqrt(sum(ifo.meta_data["optimal_SNR"] ** 2 for ifo in ifos)), 2),
)


def detector_log_likelihood(ifo, model_polarizations):
    model = ifo.get_detector_response(
        model_polarizations, source_parameters, frequencies=frequency
    )
    residual = ifo.frequency_domain_strain - model
    psd = ifo.power_spectral_density_array
    return -2 * df * np.sum(np.abs(residual[mask]) ** 2 / psd[mask])


# The chirp mass is measured to ~0.01 solar masses, so the grid must be narrow.
# A +/-2 solar mass window would be about 120 sigma wide and show only a spike.
mass_grid = np.linspace(float(theta_true[0]) - 0.1, float(theta_true[0]) + 0.1, 141)
logL_by_ifo = {ifo.name: [] for ifo in ifos}
for mc in mass_grid:
    model = polarizations(theta_true.at[0].set(mc))
    for ifo in ifos:
        logL_by_ifo[ifo.name].append(detector_log_likelihood(ifo, model))
logL_network = np.sum([logL_by_ifo[name] for name in logL_by_ifo], axis=0)


def density(logp):
    p = np.exp(logp - np.max(logp))
    return p / np.trapezoid(p, mass_grid)


log_prior_mass = np.where(
    (mass_grid >= mass_grid[0]) & (mass_grid <= mass_grid[-1]), 0.0, -np.inf
)
posterior_h1 = density(np.array(logL_by_ifo["H1"]) + log_prior_mass)
posterior_network = density(logL_network + log_prior_mass)


def summarise(density_values):
    mean = np.trapezoid(density_values * mass_grid, mass_grid)
    return np.sqrt(np.trapezoid(density_values * (mass_grid - mean) ** 2, mass_grid))


sd_h1 = summarise(posterior_h1)
sd_network = summarise(posterior_network)
snr_h1 = ifos[0].meta_data["optimal_SNR"]
snr_network = np.sqrt(sum(ifo.meta_data["optimal_SNR"] ** 2 for ifo in ifos))

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(mass_grid, posterior_h1, label=f"H1 only (SNR {snr_h1:.1f})")
ax.plot(mass_grid, posterior_network, label=f"H1+L1+V1 (SNR {snr_network:.1f})")
ax.axvline(float(theta_true[0]), color="k", ls="--", label="injection")
ax.set(
    xlabel="detector-frame chirp mass [solar masses]",
    ylabel="posterior density",
    title="A coherent network gives more information",
)
ax.legend()
plt.show()

print(f"sigma, H1 only : {sd_h1:.4f} solar masses")
print(f"sigma, network : {sd_network:.4f} solar masses")
print(f"width ratio    : {sd_h1/sd_network:.2f}")
print(f"SNR ratio      : {snr_network/snr_h1:.2f}  <- posterior width scales as 1/SNR")

### Put the same likelihood behind Bilby's interface

- A Bilby likelihood is just a class with a `log_likelihood` method and a
  declared parameter set. Nothing is hidden.
- Here Bilby wraps the **exact** network calculation written above.
- The assertion checks the library interface against the manual values, so the
  transition from hand-rolled to production code is verified, not assumed.

In [ ]:
class ChirpMassLikelihood(bilby.Likelihood):
    def __init__(self):
        super().__init__()

    def log_likelihood(self, parameters=None):
        trial_theta = theta_true.at[0].set(parameters["chirp_mass"])
        trial_polarizations = polarizations(trial_theta)
        return sum(detector_log_likelihood(ifo, trial_polarizations) for ifo in ifos)


bilby_likelihood = ChirpMassLikelihood()
bilby_priors = {
    "chirp_mass": bilby.core.prior.Uniform(
        minimum=mass_grid[0],
        maximum=mass_grid[-1],
        name="chirp_mass",
        unit="solar masses",
    )
}

bilby_log_likelihood = []
for chirp_mass in mass_grid:
    bilby_log_likelihood.append(
        bilby_likelihood.log_likelihood(parameters={"chirp_mass": chirp_mass})
    )

np.testing.assert_allclose(bilby_log_likelihood, logL_network)
print("Bilby likelihood agrees with the manual network calculation.")
print("Prior:", bilby_priors["chirp_mass"])

### A two-dimensional posterior with a real degeneracy

**Predict before running:** If we double the distance, which change in
inclination could approximately restore the observed amplitude? What feature
should that create in the joint posterior?

A one-parameter scan hides the feature that dominates real CBC results:
parameters are correlated, and some are correlated so strongly that they are
effectively measured only in combination.

The classic example is distance and inclination. For the dominant quadrupole
mode of a circular binary,

$$
h_+\propto\frac{1+\cos^2\iota}{2D_L},\qquad
h_\times\propto\frac{\cos\iota}{D_L},
$$

so both parameters enter only as amplitudes. Moving the source further away and
tilting it face-on both make the signal louder or quieter in nearly the same
way. This is why gravitational-wave distances are much less precise than
chirp masses, and why standard-siren cosmology cares so much about breaking it.

Because inclination and distance affect IMRPhenomD only through these
prefactors, we can rescale the injected polarisations instead of regenerating
the waveform, which makes a two-dimensional grid cheap. The cell asserts that
this shortcut reproduces rippleGW exactly.

Two caveats worth carrying forward. This grid uses a **flat prior on distance**
for simplicity; a real analysis uses a uniform-in-comoving-volume prior, which
grows like $D_L^2$ and therefore pushes the posterior towards larger distances.
And the posterior is one-sided in inclination here because we restricted
$\iota\le\pi/2$; the full problem is also nearly symmetric under
$\iota\rightarrow\pi-\iota$, giving the familiar two-lobed structure.

In [ ]:
true_distance, true_inclination = 800.0, 0.5


def scaled_polarizations(distance, inclination):
    """Rescale the injection to a new distance and inclination."""
    plus_ratio = ((1 + np.cos(inclination) ** 2) / 2) / (
        (1 + np.cos(true_inclination) ** 2) / 2
    )
    cross_ratio = np.cos(inclination) / np.cos(true_inclination)
    distance_ratio = true_distance / distance
    return {
        "plus": injection_polarizations["plus"] * plus_ratio * distance_ratio,
        "cross": injection_polarizations["cross"] * cross_ratio * distance_ratio,
    }


# The shortcut must agree with a full rippleGW call.
check = polarizations(ripple_parameters(distance=1300.0, inclination=0.9))
shortcut = scaled_polarizations(1300.0, 0.9)
for polarisation in ("plus", "cross"):
    np.testing.assert_allclose(
        shortcut[polarisation][mask], check[polarisation][mask], rtol=1e-10
    )
print("Amplitude rescaling reproduces rippleGW to machine precision.")

distance_grid = np.linspace(450, 1250, 70)
inclination_grid = np.linspace(0.02, np.pi / 2 - 0.02, 66)
logL_grid = np.array(
    [
        [
            sum(
                detector_log_likelihood(
                    ifo, scaled_polarizations(distance, inclination)
                )
                for ifo in ifos
            )
            for inclination in inclination_grid
        ]
        for distance in distance_grid
    ]
)

joint_posterior = np.exp(logL_grid - logL_grid.max())
joint_posterior /= np.trapezoid(
    np.trapezoid(joint_posterior, inclination_grid, axis=1), distance_grid
)
distance_marginal = np.trapezoid(joint_posterior, inclination_grid, axis=1)
inclination_marginal = np.trapezoid(joint_posterior, distance_grid, axis=0)

In [ ]:
fig = plt.figure(figsize=(9, 5.5))
grid_spec = fig.add_gridspec(
    2, 2, width_ratios=(4, 1.4), height_ratios=(1.4, 4), wspace=0.05, hspace=0.05
)
joint_ax = fig.add_subplot(grid_spec[1, 0])
top_ax = fig.add_subplot(grid_spec[0, 0], sharex=joint_ax)
side_ax = fig.add_subplot(grid_spec[1, 1], sharey=joint_ax)

joint_ax.contourf(
    distance_grid, inclination_grid, joint_posterior.T, levels=20, cmap="magma"
)
joint_ax.plot(true_distance, true_inclination, "c*", ms=14, label="injection")
joint_ax.set(xlabel="luminosity distance [Mpc]", ylabel=r"inclination $\iota$ [rad]")
joint_ax.legend(loc="upper right", facecolor="white", framealpha=0.9)

top_ax.plot(distance_grid, distance_marginal, color="C0")
top_ax.axvline(true_distance, color="k", ls="--")
top_ax.set_ylabel("marginal")
top_ax.tick_params(labelbottom=False)

side_ax.plot(inclination_marginal, inclination_grid, color="C0")
side_ax.axhline(true_inclination, color="k", ls="--")
side_ax.set_xlabel("marginal")
side_ax.tick_params(labelleft=False)
fig.suptitle("The distance-inclination degeneracy")
plt.show()


def credible_interval(grid, density, probability=0.9):
    cdf = np.r_[0, np.cumsum((density[:-1] + density[1:]) * np.diff(grid) / 2)]
    cdf /= cdf[-1]
    tail = (1 - probability) / 2
    return np.interp([tail, 0.5, 1 - tail], cdf, grid)


low, median, high = credible_interval(distance_grid, distance_marginal)
print(f"injected distance : {true_distance:.0f} Mpc")
print(f"posterior median  : {median:.0f} Mpc")
print(f"90% interval      : [{low:.0f}, {high:.0f}] Mpc")
print(
    "Fractional distance precision: "
    f"{(high - low) / (2 * median):.0%}, far worse than the chirp mass."
)

## 5. Why a network localises the sky

For detectors at positions $\mathbf x_I$ and $\mathbf x_J$, a sky direction
$\hat{\mathbf n}$ predicts

$$
\Delta t_{IJ}(\hat{\mathbf n})
=\frac{\hat{\mathbf n}\cdot(\mathbf x_I-\mathbf x_J)}{c}.
$$

- **One detector:** no time difference, so timing alone allows the whole sky.
- **Two detectors:** one measured delay selects a ring of constant
  $\Delta t_{IJ}$.
- **Three detectors:** two independent delays intersect into much smaller
  regions.

Real Bilby localisation also uses coherent phase, antenna amplitudes,
polarisation, distance-inclination correlations, waveform uncertainty, and sky
priors.

**Predict before running:** Why does one arrival-time difference make a
ring, rather than a point? When Virgo is added, which degeneracy remains because
this particular calculation still uses timing alone?

In [ ]:
ra = np.linspace(-np.pi, np.pi, 91)
dec = np.linspace(-np.pi / 2, np.pi / 2, 46)
RA, DEC = np.meshgrid(ra, dec)
delays = {
    ifo.name: np.array(
        [[ifo.time_delay_from_geocenter(r, d, gps_time) for r in ra] for d in dec]
    )
    for ifo in ifos
}
observed = {
    ifo.name: ifo.time_delay_from_geocenter(
        source_parameters["ra"], source_parameters["dec"], gps_time
    )
    for ifo in ifos
}
sigma_t = 3e-4


def timing_likelihood(names):
    # With a single detector there is no arrival-time difference to form, so
    # the timing likelihood is flat: every direction is equally allowed.
    ref = names[0]
    value = np.zeros_like(RA)
    for name in names[1:]:
        value -= (
            0.5
            * (
                (delays[name] - delays[ref] - (observed[name] - observed[ref]))
                / sigma_t
            )
            ** 2
        )
    return value


panels = [
    (["H1"], "one detector:\nno timing information"),
    (["H1", "L1"], "two detectors:\na ring of constant delay"),
    (["H1", "L1", "V1"], "three detectors:\nring intersections"),
]
fig, axes = plt.subplots(
    1, 3, figsize=(15, 3.8), subplot_kw={"projection": "mollweide"}
)
for ax, (names, title) in zip(axes, panels):
    ll = timing_likelihood(names)
    sky = np.exp(ll - ll.max())
    ax.contourf(RA, DEC, sky, levels=np.linspace(0.05, 1, 15), cmap="magma")
    ax.plot(source_parameters["ra"], source_parameters["dec"], "c*", ms=10)
    ax.set_title(title, fontsize=10)
    ax.grid(True, lw=0.4, alpha=0.5)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
plt.show()
print("Sky area allowed by timing alone shrinks with each added detector.")
print("One detector constrains direction only through its antenna pattern,")
print("which is why a single-detector alert has a nearly all-sky map.")

## 6. From events to a population

Event-level posteriors become inputs to hierarchical inference. If $\Lambda$ describes a population,
$$
p(\Lambda\mid\{d_i\},\mathrm{det})\propto p(\Lambda)
\prod_i\frac{\int p(d_i\mid\theta)p(\theta\mid\Lambda)d\theta}{\alpha(\Lambda)}.
$$
$\alpha(\Lambda)$ is the detectable fraction. Ignoring it confuses the observed catalogue with the astrophysical population.

In [ ]:
from scipy.stats import norm

population_mean, population_width = 28.0, 5.0
all_masses = rng.normal(population_mean, population_width, 8000)
all_masses = all_masses[(all_masses > 8) & (all_masses < 55)]


def detection_probability(mass):
    return 1 / (1 + np.exp(-(mass - 22) / 3.5))


detected = all_masses[rng.random(all_masses.size) < detection_probability(all_masses)][
    :40
]
mean_grid = np.linspace(18, 38, 320)
integration_grid = np.linspace(8, 55, 900)
naive = []
corrected = []
for mean in mean_grid:
    event_term = norm.logpdf(detected, mean, population_width).sum()
    alpha = np.trapezoid(
        norm.pdf(integration_grid, mean, population_width)
        * detection_probability(integration_grid),
        integration_grid,
    )
    naive.append(event_term)
    corrected.append(event_term - len(detected) * np.log(alpha))


def normalise_population(logp):
    p = np.exp(logp - np.max(logp))
    return p / np.trapezoid(p, mean_grid)


fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
mass_axis = np.linspace(8, 55, 300)
axes[0].hist(all_masses, bins=35, density=True, histtype="step", label="underlying")
axes[0].hist(detected, bins=13, density=True, alpha=0.5, label="detected")
axes[0].plot(
    mass_axis, detection_probability(mass_axis) / 20, "--", label="selection (scaled)"
)
axes[0].set(
    xlabel="mass [toy units]", ylabel="density", title="Detected is not underlying"
)
axes[0].legend()
axes[1].plot(
    mean_grid, normalise_population(np.array(naive)), label="ignores selection"
)
axes[1].plot(
    mean_grid, normalise_population(np.array(corrected)), label="selection-aware"
)
axes[1].axvline(population_mean, color="k", ls="--", label="injection")
axes[1].set(
    xlabel="population mean",
    ylabel="posterior density",
    title="Selection changes the answer",
)
axes[1].legend()
plt.show()

In [ ]:
print(f"Injected population mean: {population_mean:.2f}")
print(f"Detected-catalogue mean: {detected.mean():.2f}")
print(f"Naive MAP: {mean_grid[np.argmax(naive)]:.2f}")
print(f"Selection-aware MAP: {mean_grid[np.argmax(corrected)]:.2f}")

This compact example treats masses as exactly measured. Real population inference reweights uncertain event posteriors, estimates selection with injection campaigns, infers several hyperparameters and often the rate, and checks sensitivity to event-level priors and waveform systematics.

## 7. The full Bilby analysis

Everything so far kept one piece visible at a time. This section runs the real
thing: a production `bilby` nested-sampling analysis over four parameters,
using the same rippleGW waveform.

The pieces map onto the earlier sections exactly:

| Bilby object | What it is | Earlier section |
| --- | --- | --- |
| `WaveformGenerator` | $\theta\rightarrow(h_+,h_\times)$ | 1 |
| `Interferometer` | projection $F_+,F_\times,\Delta t$, PSD, data | 2 |
| `GravitationalWaveTransient` | $\log\mathcal L=-\frac12\sum_I(d_I-h_I\mid d_I-h_I)$ | 3, 4 |
| `PriorDict` | $\pi(\theta)$ | notebook 00 |
| `run_sampler` | nested sampling for samples **and** $\log\mathcal Z$ | notebook 00 |

Two production tricks make this fast enough to run live:

- **JIT compilation.** `jax.jit` on the rippleGW call gives a waveform in well
  under a millisecond, so the run takes a couple of minutes rather than hours.
  This mirrors bilby's own `jax_fast_tutorial.py`.
- **Analytic marginalisation.** The coalescence phase $\phi_c$ can be
  integrated out exactly,

$$
\mathcal L_{\rm marg}(d\mid\theta)=\int_0^{2\pi}
\mathcal L(d\mid\theta,\phi_c)\,\frac{d\phi_c}{2\pi}
\;\propto\; I_0\!\left(|(d\mid h)|\right),
$$

  with $I_0$ a modified Bessel function. That is one fewer sampled dimension
  for free. Distance can be marginalised the same way, but we keep it
  **sampled** so the corner plot shows the distance-inclination degeneracy
  found on a grid in Section 4.

In [ ]:
import time

import jax

# A rippleGW waveform in the form Bilby expects: theta -> (h_plus, h_cross).
jitted_waveform = jax.jit(gen_IMRPhenomD_hphc)


def ripple_bbh(
    frequency_array,
    chirp_mass,
    mass_ratio,
    luminosity_distance,
    theta_jn,
    phase,
    chi_1,
    chi_2,
    **kwargs,
):
    """Bilby frequency-domain source model backed by rippleGW IMRPhenomD."""
    minimum_frequency = kwargs.get("minimum_frequency", 20.0)
    # Evaluate in band only. Clamping the array instead would create duplicate
    # frequencies, and IMRPhenomD returns NaN for those.
    in_band = frequency_array >= minimum_frequency
    eta = mass_ratio / (1 + mass_ratio) ** 2
    theta = jnp.array(
        [chirp_mass, eta, chi_1, chi_2, luminosity_distance, 0.0, phase, theta_jn]
    )
    hp, hc = jitted_waveform(
        jnp.asarray(frequency_array[in_band]), theta, jnp.asarray(minimum_frequency)
    )
    plus = np.zeros(frequency_array.size, dtype=complex)
    cross = np.zeros(frequency_array.size, dtype=complex)
    plus[in_band] = np.asarray(hp)
    cross[in_band] = np.asarray(hc)
    return dict(plus=plus, cross=cross)


full_injection = dict(
    chirp_mass=28.1,
    mass_ratio=0.8,
    luminosity_distance=800.0,
    theta_jn=0.5,
    phase=0.3,
    chi_1=0.1,
    chi_2=-0.1,
    ra=1.2,
    dec=-0.4,
    psi=0.7,
    geocent_time=gps_time,
)

waveform_generator = bilby.gw.WaveformGenerator(
    duration=duration,
    sampling_frequency=sample_rate,
    frequency_domain_source_model=ripple_bbh,
    parameter_conversion=lambda parameters: (parameters, []),
    waveform_arguments=dict(minimum_frequency=f_min),
)

bilby.core.utils.random.seed(20260817)
full_ifos = bilby.gw.detector.InterferometerList(["H1", "L1"])
full_ifos.set_strain_data_from_power_spectral_densities(
    sampling_frequency=sample_rate, duration=duration, start_time=gps_time - 2
)
full_polarizations = waveform_generator.frequency_domain_strain(full_injection)
for ifo in full_ifos:
    ifo.inject_signal_from_waveform_polarizations(full_injection, full_polarizations)

network_snr = np.sqrt(sum(ifo.meta_data["optimal_SNR"] ** 2 for ifo in full_ifos))
print("Injected network SNR:", round(float(network_snr), 2))

In [ ]:
full_priors = bilby.core.prior.PriorDict()
# Held fixed: sky position, polarisation, arrival time, and spins.
for name in ["chi_1", "chi_2", "ra", "dec", "psi", "geocent_time"]:
    full_priors[name] = full_injection[name]
# Sampled: two mass parameters, orientation, and distance.
full_priors["chirp_mass"] = bilby.core.prior.Uniform(
    27.5, 28.7, name="chirp_mass", latex_label=r"$\mathcal{M}$"
)
full_priors["mass_ratio"] = bilby.core.prior.Uniform(
    0.3, 1.0, name="mass_ratio", latex_label="$q$"
)
full_priors["theta_jn"] = bilby.core.prior.Sine(
    name="theta_jn", latex_label=r"$\theta_{JN}$"
)
full_priors["luminosity_distance"] = bilby.gw.prior.UniformSourceFrame(
    200, 3000, name="luminosity_distance", latex_label="$d_L$"
)
# Marginalised analytically rather than sampled.
full_priors["phase"] = bilby.core.prior.Uniform(
    0, 2 * np.pi, name="phase", boundary="periodic"
)

full_likelihood = bilby.gw.likelihood.GravitationalWaveTransient(
    interferometers=full_ifos,
    waveform_generator=waveform_generator,
    priors=full_priors,
    phase_marginalization=True,
)

full_likelihood.parameters.update(full_injection)
start = time.time()
for _ in range(50):
    full_likelihood.parameters.update(full_priors.sample())
    full_likelihood.log_likelihood_ratio()
print(f"one likelihood evaluation: {(time.time() - start) / 50 * 1e3:.2f} ms")

Now the sampler. This is a genuine nested-sampling run, not a
grid, and takes a couple of minutes. It returns posterior samples **and** the
evidence.

In [ ]:
start = time.time()
full_result = bilby.run_sampler(
    likelihood=full_likelihood,
    priors=full_priors,
    sampler="dynesty",
    nlive=250,
    sample="acceptance-walk",
    naccept=15,
    injection_parameters=full_injection,
    outdir="bilby_out",
    label="fqcp_full",
    result_class=bilby.gw.result.CBCResult,
    clean=True,
    plot=False,
    save=False,
    print_progress=False,
)
print(f"sampling wall time: {time.time() - start:.0f} s")
print(f"log Bayes factor (signal vs noise): {full_result.log_bayes_factor:.1f}")
print(f"posterior samples: {len(full_result.posterior)}")

In [ ]:
sampled_names = ["chirp_mass", "mass_ratio", "theta_jn", "luminosity_distance"]
print(f"{'parameter':22s}{'median':>10s}{'90% interval':>26s}{'truth':>10s}")
for name in sampled_names:
    low, median, high = full_result.posterior[name].quantile([0.05, 0.5, 0.95])
    interval = f"[{low:.3f}, {high:.3f}]"
    print(f"{name:22s}{median:10.3f}{interval:>26s}{full_injection[name]:10.3f}")

full_result.plot_corner(
    parameters=sampled_names,
    truths=[full_injection[name] for name in sampled_names],
    save=False,
)
plt.show()

- Every truth should land inside its 90% interval. For a single
  noise realisation that is partly luck; the P-P test in notebook 00 is what
  checks calibration properly.
- The chirp mass is pinned to a fraction of a percent while the distance is
  uncertain at the tens-of-percent level. That gap is the message of Sections 1
  and 4: phase is measured precisely, amplitude is not.
- `luminosity_distance` against `theta_jn` shows the same curved degeneracy the
  grid produced in Section 4, now from a sampler that was never told about it.
- The log Bayes factor is the signal-versus-noise evidence ratio, the same
  quantity nested sampling produced in notebook 00.
- Scaling up to a real analysis means freeing sky position, spins, and time
  (about 15 parameters), which is why production runs take hours on many
  cores rather than two minutes on one.

## 8. Real data: a restricted GW150914 analysis

The controlled injection above tests the sampler against a known answer. We now
start a **new analysis** using public H1/L1 strain around GW150914 and noise
PSDs estimated from separate off-source data.

To keep this live exercise fast, we scan detector-frame chirp mass
$\mathcal M^{\rm det}$ and mass ratio $q$ with a non-spinning IMRPhenomD
template. At every grid point we maximise over a small arrival-time window and
over one complex amplitude per detector:

$$
\log\Lambda_{\rm prof}(\mathcal M^{\rm det},q)
=\frac12\sum_I\max_{\tau_I}\rho_I^2(\tau_I).
$$

The complex amplitude profiles over phase and amplitude. We then integrate the
profile likelihood over a flat $q$ grid. This is a genuine real-data
matched-filter inference, but the resulting **restricted density is not the
full LVK posterior**: profiling is not marginalisation, and independent
detector amplitudes discard coherent sky and polarisation information.

**Why the template must have spin.** Detector-frame chirp mass and effective
spin are degenerate: in the GWTC-1 samples they correlate at $r = 0.94$. A
non-spinning template slices that ridge at right angles, which manufactures a
sharp fake peak and a spurious second mode -- it looks precise and is an
artifact. So we grid $\chi_{\rm eff}$ (setting $\chi_1=\chi_2$ makes the grid
axis exactly $\chi_{\rm eff}$), weight it by the $\chi_{\rm eff}$ distribution
that LVK's isotropic spin prior induces, and marginalise. The left panel below
is that degeneracy plane, with LVK's answer marked on it.

Grid resolution is part of the physics here, not a detail: the ridge is narrow
in $q$ and $\chi_{\rm eff}$, and a coarse grid aliases it into lumps that look
like structure.

**It still will not match exactly, and should not.** We profile rather than
marginalise over each detector's amplitude, phase and time; we fix the sky
position and inclination; we force $\chi_1=\chi_2$ and allow no precession; and
we estimate the PSD off-source. What survives all that is agreement at about
half a standard deviation of the LVK posterior, with each median inside the
other's 90% interval.

Two things this notebook checked and found *not* to be the cause, which are
worth knowing because both are plausible: imposing a fully coherent likelihood
(one complex amplitude and one geocentric $t_c$, with the H1/L1 relative phase
locked by the antenna patterns) changes the answer by 0.02 $M_\odot$; and
widening the band to 20--900 Hz over 8 s makes the agreement slightly *worse*,
not better.

In [ ]:
import h5py
from pathlib import Path
from urllib.request import urlretrieve

GW150914_GPS = 1126259462.4
GWTC1_POSTERIOR_URL = (
    "https://dcc.ligo.org/public/0157/P1800370/005/GW150914_GWTC-1.hdf5"
)
cache = Path("gw150914_cache")
cache.mkdir(exist_ok=True)


def gw150914_strain(detector, start, end):
    # GWOSC provides the GW150914 event files at 4096 Hz (not 2048 Hz).
    # Include the rate in the cache key so stale data cannot be reused.
    path = cache / f"{detector}-{int(start)}-{int(end-start)}-4096Hz.hdf5"
    if path.exists():
        return TimeSeries.read(path)
    strain = TimeSeries.fetch_open_data(detector, start, end, sample_rate=4096)
    strain.write(path)
    return strain


raw = {
    ifo: gw150914_strain(ifo, GW150914_GPS - 16, GW150914_GPS + 16)
    for ifo in ("H1", "L1")
}
off_source = {
    ifo: gw150914_strain(ifo, GW150914_GPS + 32, GW150914_GPS + 160)
    for ifo in ("H1", "L1")
}
print("Downloaded/cached 32 s analysis and 128 s off-source data for H1 and L1.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, ifo in zip(axes, ("H1", "L1")):
    # Whiten first: band-passing alone leaves the 35-50 Hz noise wall, which is
    # far louder than the signal. Autoscaling also blows up on the filter's
    # ring-up at the 32 s segment edges, so fix the limits.
    filtered = raw[ifo].whiten(4, 2).bandpass(35, 300).notch(60).notch(120)
    time_from_event = filtered.times.value - GW150914_GPS
    ax.plot(time_from_event, filtered.value, lw=0.9)
    ax.set(
        xlim=(-0.25, 0.08),
        ylim=(-6, 6),
        xlabel="time from GW150914 [s]",
        title=f"{ifo}: whitened, band-passed strain",
    )
axes[0].set_ylabel("whitened strain [sigma]")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, ifo in zip(axes, ("H1", "L1")):
    q = raw[ifo].q_transform(
        outseg=(GW150914_GPS - 0.25, GW150914_GPS + 0.08),
        frange=(30, 350),
        qrange=(4, 64),
    )
    image = ax.pcolormesh(
        q.times.value - GW150914_GPS,
        q.frequencies.value,
        q.value.T,
        shading="nearest",
        cmap="magma",
    )
    ax.set(
        yscale="log",
        xlabel="time from event [s]",
        ylabel="frequency [Hz]",
        title=f"{ifo}: Q-transform",
    )
    fig.colorbar(image, ax=ax, label="normalised energy")
plt.show()

### Build the real-data likelihood

- Downsample the cached 4096-Hz strain to the notebook's 1024-Hz working rate.
- Analyse the four seconds centred on the event.
- Estimate each detector PSD from a separate 128-second segment using median
  Welch averaging.
- Search only within $\pm60$ ms of the nominal event time, wide enough for the
  H1--L1 delay and the restricted waveform's timing uncertainty.

In [ ]:
from scipy.special import logsumexp

REAL_SAMPLE_RATE = 1024
REAL_DURATION = 4.0
REAL_F_MIN = 30.0
REAL_F_MAX = 350.0
REAL_START = GW150914_GPS - REAL_DURATION / 2

real_ifos = bilby.gw.detector.InterferometerList([])
for name in ("H1", "L1"):
    analysis = (
        raw[name]
        .resample(REAL_SAMPLE_RATE)
        .crop(REAL_START, REAL_START + REAL_DURATION)
    )
    noise = off_source[name].resample(REAL_SAMPLE_RATE)
    noise_psd = noise.psd(
        fftlength=REAL_DURATION,
        overlap=REAL_DURATION / 2,
        window=("tukey", 0.2),
        method="median",
    )
    ifo = bilby.gw.detector.get_empty_interferometer(name)
    ifo.minimum_frequency = REAL_F_MIN
    ifo.maximum_frequency = REAL_F_MAX
    ifo.power_spectral_density = bilby.gw.detector.PowerSpectralDensity(
        frequency_array=noise_psd.frequencies.value,
        psd_array=noise_psd.value,
    )
    ifo.set_strain_data_from_gwpy_timeseries(analysis)
    real_ifos.append(ifo)

real_frequency = real_ifos[0].frequency_array
real_df = real_frequency[1] - real_frequency[0]
real_n_samples = int(REAL_SAMPLE_RATE * REAL_DURATION)
real_band = (real_frequency >= REAL_F_MIN) & (real_frequency <= REAL_F_MAX)
real_time_offsets = (np.arange(real_n_samples) - real_n_samples // 2) / REAL_SAMPLE_RATE
real_time_window = np.abs(real_time_offsets) <= 0.060
event_time_in_segment = GW150914_GPS - real_ifos[0].strain_data.start_time


def real_template(chirp_mass_detector, mass_ratio, aligned_spin=0.0):
    """Aligned-spin IMRPhenomD plus polarisation on the real-data grid.

    Setting chi1 = chi2 = aligned_spin makes this grid axis exactly chi_eff.
    """
    symmetric_mass_ratio = mass_ratio / (1 + mass_ratio) ** 2
    parameters = jnp.array(
        [
            chirp_mass_detector,
            symmetric_mass_ratio,
            aligned_spin,
            aligned_spin,
            1.0,
            event_time_in_segment,
            0.0,
            0.0,
        ]
    )
    plus, _ = jitted_waveform(
        jnp.asarray(real_frequency[real_band]),
        parameters,
        jnp.asarray(REAL_F_MIN),
    )
    template = np.zeros(real_frequency.size, dtype=complex)
    template[real_band] = np.asarray(plus)
    return template


def profile_detector(ifo, template, return_model=False):
    """Profile one detector over a complex amplitude and a narrow time shift."""
    psd = ifo.power_spectral_density_array
    usable = real_band & np.isfinite(psd) & (psd > 0)
    integrand = np.zeros(real_frequency.size, dtype=complex)
    integrand[usable] = (
        ifo.frequency_domain_strain[usable] * np.conj(template[usable]) / psd[usable]
    )
    padded = np.zeros(real_n_samples, dtype=complex)
    padded[: integrand.size] = integrand
    complex_overlap = np.fft.fftshift(
        4 * real_df * real_n_samples * np.fft.ifft(padded)
    )
    template_norm = 4 * real_df * np.sum(np.abs(template[usable]) ** 2 / psd[usable])
    allowed_indices = np.flatnonzero(real_time_window)
    peak_index = allowed_indices[np.argmax(np.abs(complex_overlap[real_time_window]))]
    peak_snr = np.abs(complex_overlap[peak_index]) / np.sqrt(template_norm)
    peak_time = real_time_offsets[peak_index]
    if not return_model:
        return float(peak_snr), float(peak_time)
    complex_scale = complex_overlap[peak_index] / template_norm
    shifted_template = template * np.exp(-2j * np.pi * real_frequency * peak_time)
    return float(peak_snr), float(peak_time), complex_scale * shifted_template


# Spin is not optional here. Chirp mass and chi_eff are strongly degenerate, so
# a chi = 0 template slices that ridge at right angles and manufactures a sharp
# fake peak plus a spurious second mode. Grid the spin instead, and marginalise.
# Resolution matters too: too few q/chi points and the narrow ridge aliases into
# lumps. 91 x 41 x 51 is converged (matches a 145 x 61 x 81 grid to 0.003 in
# density); drop to 41 x 21 x 25 for a faster, visibly lumpier live run.
real_mc_grid = np.linspace(29.0, 33.5, 91)
real_q_grid = np.linspace(0.4, 1.0, 41)
real_chi_grid = np.linspace(-0.6, 0.6, 51)
real_log_profile = np.empty((real_mc_grid.size, real_q_grid.size, real_chi_grid.size))
for i, chirp_mass_detector in enumerate(real_mc_grid):
    for j, mass_ratio in enumerate(real_q_grid):
        for k, aligned_spin in enumerate(real_chi_grid):
            template = real_template(chirp_mass_detector, mass_ratio, aligned_spin)
            squared_network_snr = sum(
                profile_detector(ifo, template)[0] ** 2 for ifo in real_ifos
            )
            real_log_profile[i, j, k] = 0.5 * squared_network_snr


# A flat chi prior is not what LVK used, and the difference matters on a ridge.
# Build their prior -- spin magnitudes U(0, 1), isotropic tilts -- by sampling
# the induced chi_eff distribution onto our grid.
def lvk_chi_eff_log_prior(chi_grid, q_low, q_high, n_draws=2_000_000, seed=1):
    rng = np.random.default_rng(seed)
    a1, a2 = rng.uniform(0, 1, n_draws), rng.uniform(0, 1, n_draws)
    cos1, cos2 = rng.uniform(-1, 1, n_draws), rng.uniform(-1, 1, n_draws)
    q = rng.uniform(q_low, q_high, n_draws)
    chi_eff = (a1 * cos1 + q * a2 * cos2) / (1 + q)
    half = (chi_grid[1] - chi_grid[0]) / 2
    edges = np.concatenate(
        [
            [chi_grid[0] - half],
            (chi_grid[:-1] + chi_grid[1:]) / 2,
            [chi_grid[-1] + half],
        ]
    )
    weights, _ = np.histogram(chi_eff, bins=edges, density=True)
    return np.log(np.maximum(weights, 1e-12))


real_log_chi_prior = lvk_chi_eff_log_prior(
    real_chi_grid, real_q_grid[0], real_q_grid[-1]
)

# Flat in q, LVK-like in chi_eff; marginalise over both nuisance axes.
real_log_posterior = real_log_profile + real_log_chi_prior[None, None, :]
real_log_mc_density = logsumexp(real_log_posterior, axis=(1, 2))
real_mc_density = np.exp(real_log_mc_density - real_log_mc_density.max())
real_mc_density /= np.trapezoid(real_mc_density, real_mc_grid)
real_low, real_median, real_high = credible_interval(real_mc_grid, real_mc_density)

real_log_chi_density = logsumexp(real_log_posterior, axis=(0, 1))
real_chi_density = np.exp(real_log_chi_density - real_log_chi_density.max())
real_chi_density /= np.trapezoid(real_chi_density, real_chi_grid)
real_chi_low, real_chi_median, real_chi_high = credible_interval(
    real_chi_grid, real_chi_density
)

# Marginalised (Mc, chi_eff) surface: this is the plane the degeneracy lives in.
real_log_mc_chi = logsumexp(real_log_posterior, axis=1)

best_i, best_j, best_k = np.unravel_index(
    np.argmax(real_log_profile), real_log_profile.shape
)
real_best_mc = real_mc_grid[best_i]
real_best_q = real_q_grid[best_j]
real_best_chi = real_chi_grid[best_k]
real_best_template = real_template(real_best_mc, real_best_q, real_best_chi)
real_best_snr = np.sqrt(2 * real_log_profile[best_i, best_j, best_k])

print(
    f"profile maximum: Mc_det={real_best_mc:.2f} Msun, "
    f"q={real_best_q:.2f}, chi_eff={real_best_chi:+.2f}"
)
print(
    f"restricted Mc_det density: {real_median:.2f} "
    f"[{real_low:.2f}, {real_high:.2f}] Msun"
)
print(
    f"restricted chi_eff:        {real_chi_median:+.3f} "
    f"[{real_chi_low:+.3f}, {real_chi_high:+.3f}]"
)
print(f"profiled H1+L1 network SNR: {real_best_snr:.1f}")

### Compare like with like

The release contains several posterior datasets. Select `Overall_posterior`
explicitly and use its `m1_detector_frame_Msun` and
`m2_detector_frame_Msun` fields. We therefore compare detector-frame chirp
mass with detector-frame chirp mass; no accidental source/detector-frame mixing
is allowed.

In [ ]:
posterior_path = cache / "GW150914_GWTC-1.hdf5"
if not posterior_path.exists():
    urlretrieve(GWTC1_POSTERIOR_URL, posterior_path)

with h5py.File(posterior_path, "r") as h5:
    if "Overall_posterior" not in h5:
        raise KeyError("GWTC-1 file does not contain Overall_posterior")
    lvk = h5["Overall_posterior"][...]

required_mass_fields = {"m1_detector_frame_Msun", "m2_detector_frame_Msun"}
if not required_mass_fields <= set(lvk.dtype.names or ()):
    raise KeyError(f"Missing detector-frame mass fields: {required_mass_fields}")

lvk_m1_detector = lvk["m1_detector_frame_Msun"]
lvk_m2_detector = lvk["m2_detector_frame_Msun"]
lvk_chirp_mass_detector = (lvk_m1_detector * lvk_m2_detector) ** (3 / 5) / (
    lvk_m1_detector + lvk_m2_detector
) ** (1 / 5)
lvk_low, lvk_median, lvk_high = np.percentile(lvk_chirp_mass_detector, [5, 50, 95])

# chi_eff for the like-for-like spin comparison.
lvk_chi_eff = (
    lvk_m1_detector * lvk["spin1"] * lvk["costilt1"]
    + lvk_m2_detector * lvk["spin2"] * lvk["costilt2"]
) / (lvk_m1_detector + lvk_m2_detector)
lvk_chi_low, lvk_chi_median, lvk_chi_high = np.percentile(lvk_chi_eff, [5, 50, 95])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# The degeneracy plane. LVK's answer should land on our ridge, not beside it.
image = axes[0].pcolormesh(
    real_mc_grid,
    real_chi_grid,
    (real_log_mc_chi - real_log_mc_chi.max()).T,
    vmin=-12,
    vmax=0,
    cmap="magma",
    shading="gouraud",
    rasterized=True,
)
axes[0].plot(lvk_median, lvk_chi_median, "*", color="cyan", ms=16, mec="k", mew=0.6)
axes[0].annotate(
    "LVK median",
    (lvk_median, lvk_chi_median),
    xytext=(14, -20),
    textcoords="offset points",
    color="cyan",
    fontsize=9,
    arrowprops=dict(arrowstyle="-", color="cyan", lw=0.8),
)
axes[0].set(
    xlabel=r"detector-frame chirp mass $\mathcal{M}^{\rm det}$ [$M_\odot$]",
    ylabel=r"$\chi_{\rm eff}$",
    title=r"The $\mathcal{M}$--$\chi_{\rm eff}$ degeneracy",
)
axes[0].grid(False)
fig.colorbar(image, ax=axes[0], label="log posterior rel. max")

lvk_bins = np.linspace(28.5, 34.0, 70)
axes[1].hist(
    lvk_chirp_mass_detector,
    bins=lvk_bins,
    density=True,
    histtype="step",
    lw=2,
    color="C0",
    label="LVK GWTC-1",
)
axes[1].plot(real_mc_grid, real_mc_density, lw=2.5, color="C3", label="this notebook")
axes[1].set(
    xlabel=r"detector-frame chirp mass $\mathcal{M}^{\rm det}$ [$M_\odot$]",
    ylabel="density",
    title="Chirp mass",
)
axes[1].legend(fontsize=8)

axes[2].hist(
    lvk_chi_eff,
    bins=np.linspace(-0.5, 0.5, 60),
    density=True,
    histtype="step",
    lw=2,
    color="C0",
    label="LVK GWTC-1",
)
axes[2].plot(real_chi_grid, real_chi_density, lw=2.5, color="C3", label="this notebook")
axes[2].set(xlabel=r"$\chi_{\rm eff}$", title="Effective spin", xlim=(-0.5, 0.5))
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"LVK  Mc_det  : {lvk_median:.2f} [{lvk_low:.2f}, {lvk_high:.2f}] Msun")
print(f"ours Mc_det  : {real_median:.2f} [{real_low:.2f}, {real_high:.2f}] Msun")
print(
    f"LVK  chi_eff : {lvk_chi_median:+.3f} "
    f"[{lvk_chi_low:+.3f}, {lvk_chi_high:+.3f}]"
)
print(
    f"ours chi_eff : {real_chi_median:+.3f} "
    f"[{real_chi_low:+.3f}, {real_chi_high:+.3f}]"
)
offset = abs(real_median - lvk_median) / ((lvk_high - lvk_low) / 3.29)
print(f"\nchirp-mass offset: {offset:.2f} sigma of the LVK posterior")
# Each median must sit inside the other's 90% interval.
assert lvk_low <= real_median <= lvk_high
assert real_low <= lvk_median <= real_high
assert offset < 1.0
print("like-for-like check passed")

### Frequency-domain model check

At the profile maximum, reconstruct each detector's independently profiled
waveform. Data, noise ASD, and model are plotted in the same
strain-per-square-root-Hz convention. This checks where the fitted waveform
draws its support; it is not a posterior-predictive distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, ifo in zip(axes, real_ifos):
    peak_snr, peak_time, best_model = profile_detector(
        ifo, real_best_template, return_model=True
    )
    amplitude_factor = np.sqrt(2 / REAL_DURATION)
    ax.loglog(
        real_frequency[real_band],
        amplitude_factor * np.abs(ifo.frequency_domain_strain[real_band]),
        color="0.65",
        lw=0.7,
        label="four-second data amplitude",
    )
    ax.loglog(
        real_frequency[real_band],
        np.sqrt(ifo.power_spectral_density_array[real_band]),
        color="k",
        lw=1.3,
        label="off-source ASD",
    )
    ax.loglog(
        real_frequency[real_band],
        amplitude_factor * np.abs(best_model[real_band]),
        color="C3",
        lw=1.5,
        label="profiled best-fit model",
    )
    ax.set(
        xlim=(30, 350),
        ylim=(2e-24, 3e-21),
        xlabel="frequency [Hz]",
        title=f"{ifo.name}: SNR {peak_snr:.1f}, time shift {1e3*peak_time:+.1f} ms",
    )
    tidy_log_frequency(ax, ticks=(30, 50, 100, 200, 350))
axes[0].set_ylabel(r"amplitude equivalent [strain/$\sqrt{\rm Hz}$]")
axes[0].legend(fontsize=8)
plt.show()

## Boundary and extensions

- Section 7 frees four parameters; production BBH analyses free about fifteen, plus nuisance and systematic choices.
- Section 8 uses real strain but profiles detector amplitudes, phases and times independently. It is intentionally less complete than coherent Bayesian PE.
- Real data contain PSD uncertainty, lines, glitches, non-stationarity, and calibration uncertainty; none is marginalised here.
- Replace the profile grid with a coherent Bilby likelihood to turn this into a full real-data follow-up.
- Extend the mock catalogue by giving each event a mass posterior instead of an exact mass.

Adapted substantially from `nz_bilby_cbc_workshop_2024`, with its injection → PSD → prior → likelihood → result structure.

## Question bank and answer key

1. Does adding a detector make every sky direction equally loud, and what does a
   response map omit about localisation?
2. Why can matched filtering recover an otherwise invisible signal, and why does
   waveform mismatch reduce its SNR?
3. Why do distance and inclination form an extended posterior degeneracy?
4. Why does one inter-detector time difference produce a ring instead of a
   unique sky position?

<details>
<summary>Show the answer key</summary>

1. A new detector fills some blind spots and improves the network unevenly. A
   sensitivity map omits the phase and arrival-time information that is crucial
   for localisation.
2. The correct template adds hundreds of signal cycles coherently while noise
   adds incoherently. Mismatch lets the template phase drift, so that coherent
   sum is lost.
3. Both parameters predominantly rescale the two polarisations. A more distant,
   more face-on source can resemble a nearer, more inclined source.
4. A fixed path-length difference defines a locus on the celestial sphere. A
   third timing constraint reduces that locus, but timing alone still leaves
   mirror/extended degeneracies that coherent amplitudes and phases help break.
</details>